In [ ]:
from __future__ import print_function, division
import os
import torch
import pandas as pd
from skimage import io, transform
import numpy as np
import random
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, utils

# Ignore warnings
import warnings
warnings.filterwarnings("ignore")

plt.ion()   # interactive mode

In [ ]:
checkdf = pd.read_csv("./bdf_9.csv", index_col=0)
checkdf.head()

In [ ]:
bdf = pd.read_csv("./RF_extracted_features.csv", index_col=0)
bdf.head()

In [ ]:
positives = bdf[bdf['labels']==1].index
len(positives)

In [ ]:
len(bdf[bdf['labels']!=1].index)

Now we have to pull out the 2nd, 3rd, 4th, and 5th degree.

In [ ]:
reference = list(range(positives[0], positives[108]))
i = 0
while reference[i]==positives[i]:
    i+=1

In [ ]:
np.random.choice(1,10)

In [ ]:
idxs = np.zeros((1,len(positives)))

In [ ]:
trials = checkdf['trials'].unique()
trials = sorted(trials)
trials = trials[1:]

for trial in trials:
    tdf = checkdf[(checkdf['trials']==trial) & (checkdf['stim_onset']>0)]
    bdf.loc[tdf.index[5:10], 'labels'] += 1
    bdf.loc[tdf.index[90:95], 'labels'] += 1
    bdf.loc[tdf.index[10:20], 'labels'] += 2
    bdf.loc[tdf.index[81:90], 'labels'] += 2
    bdf.loc[tdf.index[20:30], 'labels'] += 3
    bdf.loc[tdf.index[71:80], 'labels'] += 3
    bdf.loc[tdf.index[30:45], 'labels'] += 4
    bdf.loc[tdf.index[56:70], 'labels'] += 4
    bdf.loc[tdf.index[45:55], 'labels'] += 5

In [ ]:
bdf.to_csv("./RF_extracted_features_modified-labels.csv")

In [ ]:
idxs = [[1]*10,[2]*10,[3]*10,[4]*10,[5]*10,[5]*10,[4]*10,[3]*10,[2]*10,[1]*10,0]#*95

In [ ]:
idxs = [np.array(i) for i in idxs]

In [ ]:
idxs = np.array(idxs)

In [ ]:
cdf = bdf.loc[:,['xint_rolling_14', 'xint_rolling_20', 'xint_rolling_29',
               'xint_rolling_35', 'xint_rolling_41', 'yint_rolling_41',
               'xint_rolling_116', 'yint_rolling_116', 'yint_rolling_119',
               'yint_rolling_122', 'xint_rolling_125', 'yint_rolling_125',
               'xint_rolling_128', 'yint_rolling_128', 'xint_rolling_131',
               'yint_rolling_131', 'xint_rolling_134', 'yint_rolling_134',
               'xint_rolling_137', 'yint_rolling_137', 'xint_rolling_140',
               'yint_rolling_140', 'xint_rolling_143', 'yint_rolling_143',
               'xint_rolling_146', 'yint_rolling_146', 'xint_rolling_149',
               'yint_rolling_149', 'labels']]
cdf.to_csv("./RF_extracted_features.csv")

#### Data Pre-Processing and Transformation

In [ ]:
class DataCurator(Dataset):
    
    def __init__(self, csv_file, transform=None):
        '''
        CSV file of features and labels.
        '''
        
        self.df = pd.read_csv(csv_file, index_col=0)
        self.labels = self.df['labels']
        self.data = self.df.iloc[:,:-1]
        self.cols = self.df.columns[:-1]
        
        self.transform = transform
        
    def __len__(self):
        return self.df.shape[0]
    
    def get_labels(self):
        return self.labels
    
    def get_data(self):
        return self.data
    
    def get_columns(self):
        return self.cols
    

In [ ]:
class Undersample(object):
    
    def __init__(self, matrix):
        self.n_pos = len(matrix[:,-1,0][matrix[:,-1,0]>0])
        self.X = matrix
    
    def __call__(self):
        tmpx = np.ones((self.n_pos*2, self.X.shape[1], self.X.shape[2]))
        n_negative = 0
        j = 0
        for i in range(self.X.shape[0]):
            if j == tmpx.shape[0]:
                # if we run out of tmpx slots, we stop
                return tmpx

            elif self.X[i,-1,0] > 0: # if positive, we always add
                tmpx[j,:,:] = self.X[i,:,:]
                j += 1
            elif self.X[i,-1,0] == 0 & n_negative < self.n_pos: # if negative example, but count is lower than positive, we add
                if random.uniform(0,1.0) < 0.015:
                    tmpx[j,:,:] = self.X[i,:,:]
                    j += 1
                    n_negative += 0
            else:
                continue


In [ ]:
class BinData(object):
    
    def __init__(self, DC_object):
        self.data = DC_object.get_data()
        self.labels = DC_object.get_labels()
        self.cols = DC_object.get_columns()
        self.endpt = len(DC_object)
    
    def __call__(self, height):
        tmpx = np.ones((self.endpt//height, len(self.cols), height))
        self.n_pos = 0
        for i, t in enumerate(range(height,self.endpt+1,height)):
            tmpdf = self.data.iloc[t-height:t,:-1]
            tmpx[i,:len(self.cols)-1,:] = np.transpose(tmpdf.values)
            tmpx[i,len(self.cols)-1,:] = np.sum(self.labels.loc[tmpdf.index])/height
        np.random.shuffle(tmpx)
        return tmpx


In [ ]:
data = DataCurator("./RF_extracted_features.csv")

In [ ]:
data = BinData(data)
binned_data = data(25)

In [ ]:
data = Undersample(binned_data)

In [ ]:
in_matrix = data()

In [ ]:
pos = 0
neg = 0
for i in range(in_matrix.shape[0]):
    if in_matrix[i,-1,0]>0:
        pos += 1
    else:
        neg +=1

In [ ]:
print(pos, neg)

In [ ]:
train_loader = torch.utils.data.DataLoader(in_matrix, batch_size=100, shuffle=True)

In [ ]:
in_matrix